# 🎯 [K11_M01] 일정 데이터 구조화하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K11/k11_m01_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 작전기획·일정관리 (Level 2) / K11 군 일정 관리 AI |
| **미션** | 미션 01. 일정 데이터 구조화하기 (소요시간: 약 2.0시간) |
| **실습 환경** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

> **⚠️ [교육용 가상 데이터셋 원칙]**
> 본 실습의 모든 데이터(공지, 규정, 일정 등)는 보안 규정을 준수하여 **코드 내 난수 및 가상 시나리오로 생성**됩니다.
> 실제 부대 작전, 군사 기밀 및 개인 식별 정보는 일체 포함되지 않습니다.

In [1]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


실행 환경 준비 완료 (Python 3.14.10)


---

## 🎯 미션 01. 일정 데이터 구조화하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **가상 2주 일정(일과·근무·교육·휴가)을 시작·종료 시각이 있는 표로 구조화하고 형식을 검사합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **일정을 '11/02 00:00부터의 분' 단위 표로 바꾸고 종류별 건수와 합계 시간을 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `일정 표, 종류별 요약, 형식 검사 함수와 시험 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 데이터 구조화 — 일정을 일관된 구조로 정리하고 형식 오류를 잡아내는 검사를 만들었는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(일정 표, 종류별 요약, 형식 검사 함수와 시험 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [2]:
"""가상 한빛대대 2주 일정(일과·근무·교육·휴가)을 시작·종료 시각이 있는 표로 구조화하고 형식 오류를 검사한다."""
import re
import datetime as dt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 2주(11/02~11/15) 일정 자료 (실제 부대·인원·규정과 무관)
# 시각은 "일차(0=11/02 월요일) + HH:MM"으로 표기하며, 24시를 넘기면 26:00처럼 씁니다.
# ---------------------------------------------------------------------------
START = dt.date(2026, 11, 2)
DOW = "월화수목금토일"
DAY_END = "21:00"          # 개인 시간 계획은 저녁 점호(21:00) 전까지만 다룬다


def t(day, hhmm):
    h, m = map(int, hhmm.split(":"))
    return day * 1440 + h * 60 + m


def E(eid, name, kind, day, s, e, end_day=None, sub=""):
    return {"id": eid, "name": name, "kind": kind, "sub": sub, "day": day, "start": s, "end": e,
            "end_day": day if end_day is None else end_day}


def make_routine():
    ev = []
    for d in range(14):
        wd = d % 7
        if wd < 5:
            items = [("기상·점호", "fixed", "06:30", "07:00"), ("조식", "fixed", "07:00", "08:00"),
                     ("청소·정비", "fixed", "08:00", "09:00"), ("오전 일과", "work", "09:00", "12:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("오후 일과", "work", "13:00", "17:30"),
                     ("석식", "fixed", "17:30", "18:30"), ("저녁 점호", "fixed", "21:00", "21:30")]
        else:
            items = [("기상·점호", "fixed", "07:30", "08:00"), ("조식", "fixed", "08:00", "09:00"),
                     ("중식", "fixed", "12:00", "13:00"), ("석식", "fixed", "17:30", "18:30"),
                     ("저녁 점호", "fixed", "21:00", "21:30")]
            if wd == 5:
                items.insert(2, ("주말 청소·정비", "work", "09:00", "10:30"))
        for k, (nm, sub, s, e) in enumerate(items):
            ev.append(E(f"R{d:02d}{k}", nm, "routine", d, s, e, sub=sub))
    return ev


EVENTS = make_routine() + [
    # 개인 근무표 (가상): 불침번·경계근무·당직 보조
    E("D01", "불침번", "duty", 0, "26:00", "28:00", sub="불침번"),
    E("D02", "불침번", "duty", 2, "22:00", "24:00", sub="불침번"),
    E("D03", "경계근무", "duty", 3, "16:00", "18:00", sub="경계근무"),
    E("D04", "불침번", "duty", 4, "26:00", "28:00", sub="불침번"),
    E("D05", "당직 보조", "duty", 6, "08:00", "32:00", sub="당직"),
    E("D06", "불침번", "duty", 8, "22:00", "24:00", sub="불침번"),
    E("D07", "불침번", "duty", 9, "22:00", "24:00", sub="불침번"),
    # 교육·훈련 일정
    E("T01", "정훈교육", "training", 1, "14:00", "16:00"),
    E("T02", "사이버 보안 교육", "training", 2, "20:30", "21:30"),
    E("T03", "정신교육", "training", 3, "15:00", "17:00"),
    E("T04", "사격 훈련", "training", 4, "09:00", "12:00"),
    E("T05", "동계 피복 보급", "training", 4, "13:00", "17:00"),
    E("T06", "보안 점검 교육", "training", 7, "15:00", "17:00"),
    E("T07", "체력검정", "training", 8, "10:00", "12:00"),
    E("T08", "신체검사(의무실)", "training", 8, "10:30", "11:30"),
    E("T09", "독감 예방접종", "training", 9, "09:00", "10:30"),
    E("T10", "분대 전술훈련(오전)", "training", 10, "09:00", "12:00"),
    E("T11", "분대 전술훈련(오후)", "training", 10, "13:00", "17:30"),
    # 휴가 (금요일 오전부터 일요일 저녁까지)
    E("V01", "정기 휴가", "leave", 11, "09:00", "18:00", end_day=13),
]

# 근무 종류별 근무 후 최소 휴식 시간(시간) - 교육용 가상 기준
REST_RULE = {"불침번": 6, "당직": 12, "경계근무": 2}

# 개인 목표: priority 1(낮음)~5(높음), allow=하루 중 허용 시간대(분), pref=선호 시간대(분)
GOALS = [
    {"id": "G1", "name": "자격증 공부", "target": 720, "lens": [60, 90, 120], "day_cap": 180,
     "deadline": (10, "24:00"), "priority": 3, "allow": (540, 1260), "pref": (1110, 1260)},
    {"id": "G2", "name": "체력단련", "target": 300, "lens": [60, 90], "day_cap": 90,
     "deadline": (13, "24:00"), "priority": 2, "allow": (360, 1260), "pref": None},
    {"id": "G3", "name": "영어 단어 학습", "target": 240, "lens": [30, 60], "day_cap": 60,
     "deadline": (13, "24:00"), "priority": 1, "allow": (540, 1260), "pref": None},
]

# 일정 변경 공지 (가상): 원래 일정을 옮기거나 늘리거나 취소한다
CHANGES = [
    {"id": "S01", "from": "교육훈련과", "title": "사격 훈련 연기(11/06 -> 11/13)", "op": "move", "target": "T04",
     "day": 11, "start": "09:00", "end": "12:00"},
    {"id": "S02", "from": "교육훈련과", "title": "체력검정 일정 변경(11/10 -> 11/11)", "op": "move", "target": "T07",
     "day": 9, "start": "10:00", "end": "12:00"},
    {"id": "S03", "from": "교육훈련과", "title": "분대 전술훈련 오후 일정 야간 연장", "op": "extend", "target": "T11", "end": "21:00"},
    {"id": "S04", "from": "작전과", "title": "경계근무 취소", "op": "cancel", "target": "D03"},
]

KINDS = {"routine": "부대 일과", "duty": "근무", "training": "교육·훈련", "leave": "휴가"}


def validate(ev):
    """일정 한 건의 형식 오류 목록을 돌려준다(빈 목록이면 정상)."""
    errs = [f"{k} 누락" for k in ("id", "name", "kind", "day", "start", "end") if ev.get(k) in (None, "")]
    if ev.get("kind") not in KINDS:
        errs.append("종류 오류")
    for k in ("start", "end"):
        if ev.get(k) and not re.match(r"^\d{1,2}:[0-5]\d$", str(ev[k])):
            errs.append(f"{k} 시간 형식 오류")
    if not errs:
        if not 0 <= ev["day"] < 14:
            errs.append("기간(0~13일) 밖")
        elif t(ev.get("end_day", ev["day"]), ev["end"]) <= t(ev["day"], ev["start"]):
            errs.append("종료가 시작보다 빠르거나 같음")
    return errs


def to_df(raw):
    """일정 목록을 '시작·종료 = 11/02 00:00부터의 분' 표로 바꾼다."""
    rows = []
    for ev in raw:
        s, e = t(ev["day"], ev["start"]), t(ev.get("end_day", ev["day"]), ev["end"])
        rows.append({"id": ev["id"], "name": ev["name"], "kind": ev["kind"], "sub": ev.get("sub", ""),
                     "start": s, "end": e, "dur": e - s})
    return pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)


def fmt(x):
    d, r = divmod(int(x), 1440)
    day = START + dt.timedelta(days=d)
    return f"{day.month:02d}/{day.day:02d}({DOW[d % 7]}) {r // 60:02d}:{r % 60:02d}"

df = to_df(EVENTS)
df["시간(h)"] = (df["dur"] / 60).round(1)
print(df.assign(시작=df["start"].map(fmt), 종료=df["end"].map(fmt))[["id", "name", "kind", "시작", "종료", "시간(h)"]]
      .query("kind != 'routine'").head(12).to_string(index=False))

print(f"\n전체 일정 {len(df)}건 (2주 = 14일)")
cnt = df.groupby("kind").agg(건수=("id", "count"), 합계시간=("dur", lambda x: round(x.sum() / 60, 1)))
print(cnt.rename(index=KINDS).to_string())

errors = {e["id"]: validate(e) for e in EVENTS}
print("\n형식 검사 오류:", sum(len(v) for v in errors.values()), "건")

# 오류 데이터로 검사기 시험
bad = [
    E("X1", "시간 거꾸로", "duty", 2, "10:00", "09:00"),
    E("X2", "종류 오류", "회의", 2, "10:00", "11:00"),
    {"id": "X3", "name": "시간 형식 오류", "kind": "training", "day": 3, "start": "9시", "end": "11:00"},
    E("X4", "기간 밖", "training", 20, "09:00", "10:00"),
    E("X5", "", "training", 3, "09:00", "10:00"),
]
for b in bad:
    print(f"오류 주입 {b['id']}:", validate(b))


 id      name     kind             시작             종료  시간(h)
D01       불침번     duty 11/03(화) 02:00 11/03(화) 04:00    2.0
T01      정훈교육 training 11/03(화) 14:00 11/03(화) 16:00    2.0
T02 사이버 보안 교육 training 11/04(수) 20:30 11/04(수) 21:30    1.0
D02       불침번     duty 11/04(수) 22:00 11/05(목) 00:00    2.0
T03      정신교육 training 11/05(목) 15:00 11/05(목) 17:00    2.0
D03      경계근무     duty 11/05(목) 16:00 11/05(목) 18:00    2.0
T04     사격 훈련 training 11/06(금) 09:00 11/06(금) 12:00    3.0
T05  동계 피복 보급 training 11/06(금) 13:00 11/06(금) 17:00    4.0
D04       불침번     duty 11/07(토) 02:00 11/07(토) 04:00    2.0
D05     당직 보조     duty 11/08(일) 08:00 11/09(월) 08:00   24.0
T06  보안 점검 교육 training 11/09(월) 15:00 11/09(월) 17:00    2.0
T07      체력검정 training 11/10(화) 10:00 11/10(화) 12:00    2.0

전체 일정 121건 (2주 = 14일)
        건수   합계시간
kind             
근무       7   36.0
휴가       1   57.0
부대 일과  102  144.0
교육·훈련   11   26.0

형식 검사 오류: 0 건
오류 주입 X1: ['종료가 시작보다 빠르거나 같음']
오류 주입 X2: ['종류 오류']
오류 주입 X3: ['start 시간 형식